# AcuDock QuickDock - Molecular Docking Pipeline

**Single-click molecular docking with an interactive notebook interface.**

A self-contained notebook that provides an interactive widget-based interface for:
- Fetching & preparing protein targets from PDB
- Preparing ligands from SMILES strings
- Running molecular docking (Vina CPU or Uni-Dock GPU)
- Interactive 3D visualization of binding poses
- Batch screening of compound libraries
- Multi-protein docking (1 ligand x N proteins)

## Getting Started

1. **Run the install cell below** (takes ~2-3 minutes)
2. The runtime will **automatically restart** — this is normal
3. After restart, **skip the install cell** and run the remaining cells
4. The interactive interface will appear — interact with it directly!

**No GPU required** for Vina mode. Enable GPU runtime for Uni-Dock acceleration.

---

**License:** MIT | **Platform:** Google Colab | **Author:** AcuDock Project

In [ ]:
# === Step 1: Install Dependencies ===
# After this cell completes, the runtime will restart automatically.
# Once it restarts, SKIP this cell and run the cells below.

!pip install -q vina meeko gemmi rdkit prody py3Dmol openbabel-wheel pdbfixer pandas numpy scipy ipywidgets

# Optional: Install Uni-Dock for GPU-accelerated docking (requires GPU runtime)
# Uncomment the next line for 1000x+ speedup on NVIDIA GPUs (compute capability >= 7.0):
# !wget -q https://github.com/dptech-corp/Uni-Dock/releases/download/1.1.0/unidock-1.1.0-cuda120-linux-x86_64 -O /usr/local/bin/unidock && chmod +x /usr/local/bin/unidock && echo "Uni-Dock GPU v1.1.0 installed" || echo "Uni-Dock download failed"

# Clone AcuDock repo for shared utilities
!git clone https://github.com/Grimlock5310/AcuDock.git /content/AcuDock 2>/dev/null || (cd /content/AcuDock && git pull)

# Restart runtime so C-extension packages are loadable
import os
os.kill(os.getpid(), 9)

## Launch AcuDock QuickDock

Run the cells below to start the interactive docking interface.
An ipywidgets interface will appear with tabs for single docking, batch screening, and multi-protein docking.

In [ ]:
# === Step 2: Imports and Setup ===
import warnings
warnings.filterwarnings('ignore')

import os, sys, io
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Draw
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

sys.path.insert(0, '/content/AcuDock')
import acudock_utils as utils

WORK_DIR = '/content/acudock_quickdock'
os.makedirs(WORK_DIR, exist_ok=True)

print('Imports successful.')
print(utils.get_docking_engine_status())

In [ ]:
# === Step 3: Launch Interactive Interface ===
import ipywidgets as widgets
from IPython.display import display, HTML, FileLink, clear_output
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# Pipeline functions (print-based progress — works reliably in Colab)
# ---------------------------------------------------------------------------

def run_single_docking(pdb_id, smiles, lig_name, engine, exhaustiveness,
                        n_poses, box_size, residues_str, output_area):
    """Full single-ligand docking pipeline."""
    with output_area:
        clear_output(wait=True)
        print('Starting single docking...')

    try:
        pdb_id = pdb_id.strip().upper()
        smiles = smiles.strip()
        lig_name = lig_name.strip() or 'ligand'

        if not pdb_id:
            with output_area:
                print('Error: Please enter a PDB ID.')
            return
        if not smiles or Chem.MolFromSmiles(smiles) is None:
            with output_area:
                print('Error: Invalid SMILES string.')
            return

        with output_area:
            print(f'[1/5] Preparing protein {pdb_id}...')
        protein_pdb = utils.prepare_protein(pdb_id, output_dir=WORK_DIR)
        receptor_pdbqt = utils.pdb_to_pdbqt(protein_pdb)

        with output_area:
            print(f'[2/5] Preparing ligand {lig_name}...')
        ligand_pdbqt, lig_mol = utils.prepare_ligand(
            smiles, name=lig_name, output_dir=WORK_DIR
        )
        props = utils.get_ligand_properties(smiles)
        with output_area:
            print(f'       MW={props["MW"]} LogP={props["LogP"]} HBD={props["HBD"]} HBA={props["HBA"]}')

        with output_area:
            print('[3/5] Defining search box...')
        residues = None
        if residues_str.strip():
            residues = [int(r.strip()) for r in residues_str.split(',') if r.strip()]
        center = utils.get_binding_site_center(protein_pdb, chain='A', residues=residues)
        box = [int(box_size)] * 3
        with output_area:
            print(f'       Center: [{center[0]:.1f}, {center[1]:.1f}, {center[2]:.1f}], size={box}')

        with output_area:
            print(f'[4/5] Running Vina docking (exhaustiveness={int(exhaustiveness)})...')
            print('       This may take 1-5 minutes...')
        _, energies_raw, poses_path = utils.run_vina(
            receptor_pdbqt, ligand_pdbqt,
            center=center, box_size=box,
            exhaustiveness=int(exhaustiveness), n_poses=int(n_poses)
        )
        energies = [(e[0], e[1], e[2]) for e in energies_raw]

        with output_area:
            print(f'[5/5] Building results ({len(energies)} poses)...')

        R, T = 1.987e-3, 298.15
        results_df = pd.DataFrame({
            'Pose': range(1, len(energies) + 1),
            'Score (kcal/mol)': [e[0] for e in energies],
            'RMSD_lb (A)': [round(e[1], 2) for e in energies],
            'RMSD_ub (A)': [round(e[2], 2) for e in energies],
            'Est. Kd (uM)': [round(np.exp(e[0] / (R * T)) * 1e6, 4) for e in energies],
        })

        # 3D viewer will be created at display time

        csv_path = os.path.join(WORK_DIR, f'{lig_name}_{pdb_id}_results.csv')
        results_df.to_csv(csv_path, index=False)

        mol_2d = Chem.MolFromSmiles(smiles)
        img = Draw.MolToImage(mol_2d, size=(300, 200))
        img_buf = io.BytesIO()
        img.save(img_buf, format='PNG')
        img_buf.seek(0)
        img_path = os.path.join(WORK_DIR, f'{lig_name}_2d.png')
        with open(img_path, 'wb') as f:
            f.write(img_buf.read())

        with output_area:
            clear_output(wait=True)
            print(f'=== DOCKING COMPLETE ===')
            print(f'Protein: {pdb_id} | Ligand: {lig_name}')
            print(f'MW={props["MW"]} LogP={props["LogP"]} HBD={props["HBD"]} HBA={props["HBA"]}')
            print(f'Search box: [{center[0]:.1f}, {center[1]:.1f}, {center[2]:.1f}], size={box}')
            print(f'{len(energies)} poses | Best: {energies[0][0]:.2f} kcal/mol')
            print(f'Interpretation: {utils.score_interpretation(energies[0][0])}')
            print()
            display(HTML('<h4>2D Structure</h4>'))
            from IPython.display import Image as IPyImage
            display(IPyImage(filename=img_path))
            print()
            display(HTML('<h4>Docking Results</h4>'))
            display(results_df)
            print()
            display(FileLink(csv_path, result_html_prefix='Download CSV: '))
            print()
            display(HTML('<h4>3D Viewer — Top Pose</h4>'))
            _v_out = widgets.Output()
            display(_v_out)
            with _v_out:
                view = utils.visualize_pose(protein_pdb, poses_path, pose_index=0)
                view.show()

    except Exception as e:
        with output_area:
            print(f'\nERROR: {str(e)}')
            import traceback
            traceback.print_exc()


def run_batch_screening(pdb_id, compounds_text, engine, exhaustiveness,
                         box_size, residues_str, output_area):
    """Batch screening pipeline."""
    with output_area:
        clear_output(wait=True)
        print('Starting batch screening...')

    try:
        pdb_id = pdb_id.strip().upper()
        if not pdb_id:
            with output_area:
                print('Error: Please enter a PDB ID.')
            return

        compounds = []
        for line in compounds_text.strip().split('\n'):
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split(',', 1) if ',' in line else line.split('\t', 1)
            if len(parts) == 2:
                name, smi = parts[0].strip(), parts[1].strip()
            else:
                smi = parts[0].strip()
                name = f'Compound_{len(compounds)+1}'
            if Chem.MolFromSmiles(smi) is not None:
                compounds.append((name, smi))

        if not compounds:
            with output_area:
                print('Error: No valid compounds found.')
            return

        with output_area:
            print(f'Parsed {len(compounds)} valid compounds.')
            print(f'Preparing protein {pdb_id}...')
        protein_pdb = utils.prepare_protein(pdb_id, output_dir=WORK_DIR)
        receptor_pdbqt = utils.pdb_to_pdbqt(protein_pdb)

        residues = None
        if residues_str.strip():
            residues = [int(r.strip()) for r in residues_str.split(',') if r.strip()]
        center = utils.get_binding_site_center(protein_pdb, chain='A', residues=residues)
        box = [int(box_size)] * 3

        use_unidock = 'Uni-Dock' in engine and utils.check_unidock_available()
        batch_exh = int(exhaustiveness)

        with output_area:
            print('Preparing ligands...')
        prepared = []
        for i, (name, smi) in enumerate(compounds):
            try:
                lig_pdbqt, _ = utils.prepare_ligand(smi, name=name, output_dir=WORK_DIR)
                prepared.append((name, smi, lig_pdbqt))
            except Exception as e:
                prepared.append((name, smi, None))
        with output_area:
            print(f'{len([p for p in prepared if p[2]])} ligands prepared.')

        batch_scores = {}
        valid_pdbqts = [p[2] for p in prepared if p[2]]

        if use_unidock and len(valid_pdbqts) >= 2:
            with output_area:
                print(f'Running Uni-Dock GPU batch ({len(valid_pdbqts)} ligands)...')
            try:
                ud_results = utils.run_unidock(
                    receptor_pdbqt, valid_pdbqts,
                    center=center, box_size=box,
                    exhaustiveness=batch_exh, num_modes=5,
                    output_dir=WORK_DIR
                )
                for basename, scores in ud_results.items():
                    if scores and scores[0][0] < 0:
                        batch_scores[basename] = scores[0][0]
                with output_area:
                    print(f'Uni-Dock: {len(batch_scores)} valid results.')
            except Exception as e:
                with output_area:
                    print(f'Uni-Dock failed: {e}. Falling back to Vina.')

        remaining = [(n, s, p) for n, s, p in prepared
                     if p and os.path.basename(p) not in batch_scores]
        for i, (name, smi, lig_pdbqt) in enumerate(remaining):
            with output_area:
                print(f'Vina {i+1}/{len(remaining)}: {name}...')
            try:
                _, en, _ = utils.run_vina(
                    receptor_pdbqt, lig_pdbqt,
                    center=center, box_size=box,
                    exhaustiveness=batch_exh, n_poses=5
                )
                batch_scores[os.path.basename(lig_pdbqt)] = en[0][0] if len(en) > 0 else None
            except Exception:
                batch_scores[os.path.basename(lig_pdbqt)] = None

        R, T = 1.987e-3, 298.15
        results = []
        for name, smi, lig_pdbqt in prepared:
            best_score = batch_scores.get(os.path.basename(lig_pdbqt) if lig_pdbqt else '', None)
            mol_props = utils.get_ligand_properties(smi)
            est_kd = round(np.exp(best_score / (R * T)) * 1e6, 4) if best_score else None
            results.append({
                'Rank': 0, 'Name': name, 'SMILES': smi,
                'Score (kcal/mol)': round(best_score, 2) if best_score else None,
                'Est. Kd (uM)': est_kd,
                'MW': mol_props.get('MW'), 'LogP': mol_props.get('LogP'),
                'Interpretation': utils.score_interpretation(best_score) if best_score else 'Failed',
            })

        batch_df = pd.DataFrame(results)
        batch_df = batch_df.sort_values('Score (kcal/mol)', ascending=True).reset_index(drop=True)
        batch_df['Rank'] = range(1, len(batch_df) + 1)
        cols = ['Rank'] + [c for c in batch_df.columns if c != 'Rank']
        batch_df = batch_df[cols]

        csv_path = os.path.join(WORK_DIR, f'batch_{pdb_id}_results.csv')
        batch_df.to_csv(csv_path, index=False)

        valid = batch_df.dropna(subset=['Score (kcal/mol)'])
        fig, ax = plt.subplots(figsize=(10, max(3, len(valid) * 0.4)))
        colors = ['#2ecc71' if s <= -7 else '#f39c12' if s <= -5 else '#e74c3c'
                  for s in valid['Score (kcal/mol)']]
        ax.barh(valid['Name'], valid['Score (kcal/mol)'], color=colors, edgecolor='black', linewidth=0.5)
        ax.set_xlabel('Docking Score (kcal/mol)')
        ax.set_title(f'Batch Screening: {pdb_id}')
        ax.axvline(x=-7, color='gray', linestyle='--', alpha=0.5, label='Moderate threshold')
        ax.legend()
        ax.invert_yaxis()
        plt.tight_layout()
        chart_path = os.path.join(WORK_DIR, 'batch_chart.png')
        fig.savefig(chart_path, dpi=120, bbox_inches='tight')
        plt.close(fig)

        with output_area:
            clear_output(wait=True)
            print(f'=== BATCH COMPLETE ===')
            print(f'Protein: {pdb_id} | {len(compounds)} compounds')
            print(f'Best: {batch_df["Score (kcal/mol)"].min():.2f} kcal/mol')
            print()
            display(batch_df)
            print()
            from IPython.display import Image as IPyImage
            display(IPyImage(filename=chart_path))
            print()
            display(FileLink(csv_path, result_html_prefix='Download CSV: '))

    except Exception as e:
        with output_area:
            print(f'\nERROR: {str(e)}')
            import traceback
            traceback.print_exc()


def run_multi_protein(smiles, lig_name, pdb_text, box_size, exhaustiveness,
                       output_area):
    """Multi-protein docking pipeline."""
    with output_area:
        clear_output(wait=True)
        print('Starting multi-protein docking...')

    try:
        smiles = smiles.strip()
        lig_name = lig_name.strip() or 'ligand'

        if not smiles or Chem.MolFromSmiles(smiles) is None:
            with output_area:
                print('Error: Invalid SMILES string.')
            return

        pdb_ids = []
        residues_map = {}
        for line in pdb_text.strip().split('\n'):
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            if ':' in line:
                pdb_part, res_part = line.split(':', 1)
                pdb_id = pdb_part.strip().upper()
                res_list = [int(r.strip()) for r in res_part.split(',') if r.strip()]
                if res_list:
                    residues_map[pdb_id] = res_list
            else:
                pdb_id = line.strip().upper()
            if pdb_id:
                pdb_ids.append(pdb_id)

        if not pdb_ids:
            with output_area:
                print('Error: No PDB IDs found.')
            return

        with output_area:
            print(f'Docking {lig_name} against {len(pdb_ids)} proteins...')

        def progress_cb(current, total, pdb_id, status):
            with output_area:
                print(f'  [{current+1}/{total}] {pdb_id}: {status}')

        results_df, best_pdb, best_pdb_path, best_poses = utils.dock_multi_protein(
            smiles, pdb_ids, name=lig_name,
            residues_map=residues_map,
            box_size=int(box_size),
            exhaustiveness=int(exhaustiveness),
            output_dir=WORK_DIR,
            progress_callback=progress_cb
        )

        csv_path = os.path.join(WORK_DIR, f'{lig_name}_multi_protein.csv')
        results_df.to_csv(csv_path, index=False)

        valid = results_df.dropna(subset=['Best_Score'])
        fig, ax = plt.subplots(figsize=(10, max(3, len(valid) * 0.5)))
        colors = ['#2ecc71' if s <= -7 else '#f39c12' if s <= -5 else '#e74c3c'
                  for s in valid['Best_Score']]
        ax.barh(valid['PDB_ID'], valid['Best_Score'], color=colors,
                edgecolor='black', linewidth=0.5)
        ax.set_xlabel('Docking Score (kcal/mol)')
        ax.set_title(f'{lig_name} vs Multiple Proteins')
        ax.axvline(x=-7, color='gray', linestyle='--', alpha=0.5, label='Moderate threshold')
        ax.legend()
        ax.invert_yaxis()
        plt.tight_layout()
        chart_path = os.path.join(WORK_DIR, 'multi_protein_chart.png')
        fig.savefig(chart_path, dpi=120, bbox_inches='tight')
        plt.close(fig)

        has_viewer = best_pdb and best_pdb_path and best_poses

        with output_area:
            clear_output(wait=True)
            print(f'=== MULTI-PROTEIN DOCKING COMPLETE ===')
            print(f'{lig_name} docked against {len(pdb_ids)} proteins')
            if best_pdb:
                best_row = results_df[results_df['PDB_ID'] == best_pdb].iloc[0]
                print(f'Best target: {best_pdb} ({best_row["Best_Score"]:.2f} kcal/mol)')
            print()
            display(results_df[['PDB_ID', 'Best_Score', 'Est_Kd_uM', 'Interpretation',
                                'Num_Poses', 'Prep_Time_s', 'Dock_Time_s', 'Error']])
            print()
            from IPython.display import Image as IPyImage
            display(IPyImage(filename=chart_path))
            display(FileLink(csv_path, result_html_prefix='Download CSV: '))
            if has_viewer:
                print()
                display(HTML(f'<h4>Best Target: {best_pdb}</h4>'))
                _v_out = widgets.Output()
                display(_v_out)
                with _v_out:
                    view = utils.visualize_pose(best_pdb_path, best_poses, pose_index=0)
                    view.show()

    except Exception as e:
        with output_area:
            print(f'\nERROR: {str(e)}')
            import traceback
            traceback.print_exc()


# ---------------------------------------------------------------------------
# Build ipywidgets Interface
# ---------------------------------------------------------------------------

DEFAULT_COMPOUNDS = """Aspirin, CC(=O)Oc1ccccc1C(=O)O
Ibuprofen, CC(C)Cc1ccc(cc1)C(C)C(=O)O
Caffeine, Cn1c(=O)c2c(ncn2C)n(C)c1=O
Acetaminophen, CC(=O)Nc1ccc(O)cc1
Naproxen, COc1ccc2cc(ccc2c1)C(C)C(=O)O
Metformin, CN(C)C(=N)NC(=N)N
Celecoxib, Cc1ccc(-c2cc(C(F)(F)F)nn2-c2ccc(S(N)(=O)=O)cc2)cc1
Diclofenac, OC(=O)Cc1ccccc1Nc1c(Cl)cccc1Cl"""

style = {'description_width': '180px'}
layout_input = widgets.Layout(width='95%')

# === Single Docking Tab ===
sd_pdb = widgets.Text(value='1HSG', description='PDB ID:', style=style, layout=layout_input)
sd_smiles = widgets.Textarea(
    value='CC(C)(C)NC(=O)[C@@H]1C[C@@H]2CCCN2C(=O)[C@H](CC2=CC=CC=C2)NC(=O)[C@@H](CC2=CC=C(O)C=C2)N1',
    description='Ligand SMILES:', style=style, layout=widgets.Layout(width='95%', height='60px')
)
sd_name = widgets.Text(value='Indinavir', description='Ligand Name:', style=style, layout=layout_input)
sd_engine = widgets.RadioButtons(options=['Vina (CPU)', 'Uni-Dock (GPU)'], value='Vina (CPU)',
                                  description='Engine:', style=style)
sd_exhaust = widgets.IntSlider(value=8, min=8, max=128, step=8,
                                description='Exhaustiveness:', style=style, layout=layout_input)
sd_poses = widgets.IntSlider(value=20, min=5, max=50, step=5,
                              description='Max Poses:', style=style, layout=layout_input)
sd_box = widgets.IntSlider(value=20, min=15, max=40, step=5,
                            description='Box Size (A):', style=style, layout=layout_input)
sd_residues = widgets.Text(value='23,24,25,26,27,28,29,30',
                            description='Active Site Residues:', style=style,
                            layout=layout_input,
                            placeholder='Comma-separated, or blank for centroid')
sd_btn = widgets.Button(description='Run Docking', button_style='primary',
                         layout=widgets.Layout(width='95%', height='40px'))
sd_output = widgets.Output(layout=widgets.Layout(width='100%', min_height='300px',
                                                  border='1px solid #ddd'))

def on_single_dock(btn):
    btn.disabled = True
    btn.description = 'Running...'
    try:
        run_single_docking(sd_pdb.value, sd_smiles.value, sd_name.value,
                           sd_engine.value, sd_exhaust.value, sd_poses.value,
                           sd_box.value, sd_residues.value, sd_output)
    finally:
        btn.disabled = False
        btn.description = 'Run Docking'

sd_btn.on_click(on_single_dock)

single_tab = widgets.HBox([
    widgets.VBox([
        widgets.HTML('<h3>Target & Ligand</h3>'),
        sd_pdb, sd_smiles, sd_name,
        widgets.HTML('<h3>Docking Parameters</h3>'),
        sd_engine, sd_exhaust, sd_poses, sd_box, sd_residues,
        sd_btn,
    ], layout=widgets.Layout(width='40%', padding='10px')),
    widgets.VBox([sd_output],
                 layout=widgets.Layout(width='60%', padding='10px'))
])

# === Batch Screening Tab ===
bs_pdb = widgets.Text(value='1HSG', description='PDB ID:', style=style, layout=layout_input)
bs_residues = widgets.Text(value='23,24,25,26,27,28,29,30',
                            description='Active Site Residues:', style=style, layout=layout_input)
bs_engine = widgets.RadioButtons(options=['Vina (CPU)', 'Uni-Dock (GPU)'], value='Vina (CPU)',
                                  description='Engine:', style=style)
bs_exhaust = widgets.IntSlider(value=8, min=8, max=64, step=8,
                                description='Exhaustiveness:', style=style, layout=layout_input)
bs_box = widgets.IntSlider(value=20, min=15, max=40, step=5,
                            description='Box Size (A):', style=style, layout=layout_input)
bs_compounds = widgets.Textarea(value=DEFAULT_COMPOUNDS,
                                 description='Compounds:', style=style,
                                 layout=widgets.Layout(width='95%', height='200px'),
                                 placeholder='Name, SMILES (one per line)')
bs_btn = widgets.Button(description='Run Batch Screening', button_style='primary',
                         layout=widgets.Layout(width='95%', height='40px'))
bs_output = widgets.Output(layout=widgets.Layout(width='100%', min_height='300px',
                                                  border='1px solid #ddd'))

def on_batch(btn):
    btn.disabled = True
    btn.description = 'Running...'
    try:
        run_batch_screening(bs_pdb.value, bs_compounds.value, bs_engine.value,
                            bs_exhaust.value, bs_box.value, bs_residues.value,
                            bs_output)
    finally:
        btn.disabled = False
        btn.description = 'Run Batch Screening'

bs_btn.on_click(on_batch)

batch_tab = widgets.HBox([
    widgets.VBox([
        widgets.HTML('<h3>Batch Configuration</h3>'),
        bs_pdb, bs_residues, bs_engine, bs_exhaust, bs_box, bs_compounds,
        bs_btn,
    ], layout=widgets.Layout(width='40%', padding='10px')),
    widgets.VBox([bs_output],
                 layout=widgets.Layout(width='60%', padding='10px'))
])

# === Multi-Protein Tab ===
mp_smiles = widgets.Textarea(
    value='CC(C)(C)NC(=O)[C@@H]1C[C@@H]2CCCN2C(=O)[C@H](CC2=CC=CC=C2)NC(=O)[C@@H](CC2=CC=C(O)C=C2)N1',
    description='Ligand SMILES:', style=style,
    layout=widgets.Layout(width='95%', height='60px')
)
mp_name = widgets.Text(value='Indinavir', description='Ligand Name:', style=style, layout=layout_input)
mp_pdbs = widgets.Textarea(
    value='1HSG:23,24,25,26,27,28,29,30\n4LDE\n6LU7:41,49,142,144,145,163,166',
    description='PDB IDs:', style=style,
    layout=widgets.Layout(width='95%', height='120px'),
    placeholder='One PDB per line. Optional residues: 1HSG:23,24,25'
)
mp_box = widgets.IntSlider(value=20, min=15, max=40, step=5,
                            description='Box Size (A):', style=style, layout=layout_input)
mp_exhaust = widgets.IntSlider(value=32, min=8, max=128, step=8,
                                description='Exhaustiveness:', style=style, layout=layout_input)
mp_btn = widgets.Button(description='Run Multi-Protein Docking', button_style='primary',
                         layout=widgets.Layout(width='95%', height='40px'))
mp_output = widgets.Output(layout=widgets.Layout(width='100%', min_height='300px',
                                                  border='1px solid #ddd'))

def on_multi_protein(btn):
    btn.disabled = True
    btn.description = 'Running...'
    try:
        run_multi_protein(mp_smiles.value, mp_name.value, mp_pdbs.value,
                          mp_box.value, mp_exhaust.value, mp_output)
    finally:
        btn.disabled = False
        btn.description = 'Run Multi-Protein Docking'

mp_btn.on_click(on_multi_protein)

multi_tab = widgets.HBox([
    widgets.VBox([
        widgets.HTML('<h3>Ligand</h3>'),
        mp_smiles, mp_name,
        widgets.HTML('<h3>Target Proteins</h3>'),
        mp_pdbs,
        widgets.HTML('<small>Format: one PDB ID per line. Optionally add residues after a colon.<br>'
                     'Example: <code>1HSG:23,24,25,26</code></small>'),
        widgets.HTML('<h3>Parameters</h3>'),
        mp_box, mp_exhaust,
        mp_btn,
    ], layout=widgets.Layout(width='40%', padding='10px')),
    widgets.VBox([mp_output],
                 layout=widgets.Layout(width='60%', padding='10px'))
])

# === About Tab ===
about_tab = widgets.HTML(value="""
<div style="max-width:800px; padding:20px; font-family:sans-serif;">
<h3>AcuDock QuickDock</h3>
<p>A streamlined molecular docking pipeline for computational drug discovery.</p>

<h4>Pipeline</h4>
<pre>
PDB ID --> PDBFixer --> PDBQT (receptor)
SMILES --> RDKit 3D --> Meeko --> PDBQT (ligand)
                                   |
               Vina / Uni-Dock Docking
                                   |
                          Ranked Poses + 3D Viewer
</pre>

<h4>Score Interpretation</h4>
<table border="1" cellpadding="5" style="border-collapse:collapse;">
<tr><th>Score (kcal/mol)</th><th>Binding</th><th>Approx. Kd</th></tr>
<tr><td>&gt; -5</td><td>Very weak</td><td>&gt; 100 uM</td></tr>
<tr><td>-5 to -7</td><td>Moderate</td><td>10-100 uM</td></tr>
<tr><td>-7 to -9</td><td>Good</td><td>100 nM - 10 uM</td></tr>
<tr><td>&lt; -9</td><td>Strong</td><td>&lt; 100 nM</td></tr>
</table>

<h4>Docking Engines</h4>
<ul>
<li><b>Vina (CPU):</b> AutoDock Vina, 90.2% CASF docking power. No GPU needed.</li>
<li><b>Uni-Dock (GPU):</b> 1000x+ speedup for batch screening (10+ compounds).</li>
</ul>

<h4>Tips</h4>
<ul>
<li>Validate by redocking a co-crystallized ligand (RMSD &lt; 2A = success)</li>
<li>Preparation quality matters more than algorithm choice</li>
<li>Vina scores have ~2 kcal/mol error margin</li>
</ul>

<p><em>MIT License | AcuDock Project</em></p>
</div>
""")

# === Assemble Tabs ===
tabs = widgets.Tab(children=[single_tab, batch_tab, multi_tab, about_tab])
tabs.set_title(0, 'Single Docking')
tabs.set_title(1, 'Batch Screening')
tabs.set_title(2, 'Multi-Protein')
tabs.set_title(3, 'About')

display(widgets.HTML('<h1>AcuDock QuickDock</h1>'
                     '<p><b>Molecular docking made simple.</b> '
                     'Enter a protein PDB ID and a ligand SMILES to dock.</p>'))
display(tabs)
